# Preprocessing for TRT (gctg-clean)

This notebook prepares total reading time (TRT) per word AOI per subject using the clean dataset, following the lecturer's preprocessing steps. It:
- Uses the clean dataset definition and paths (no raw or practice data)
- Runs IDT event detection with pymovements defaults
- Maps fixations to word AOIs from the clean package
- Computes TRT per AOI per subject (including zeros for non-fixated AOIs)
- Adds experimental condition (neg/pos/zero) from stimulus names
- Saves tidy output to data-clean/processed/trt_by_word.csv and a parquet cache


In [1]:
# Install dependencies for reproducibility
%pip install -q pymovements polars matplotlib pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import re
import polars as pl
import pymovements as pm

BASE = Path("data-clean")
PROCESSED_DIR = BASE / "processed"
CACHE_DIR = PROCESSED_DIR / "cache"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Helpers

def is_practice(name: str) -> bool:
    return "practice" in name.lower()


def extract_condition(name: str) -> str:
    lname = name.lower()
    if "neg" in lname:
        return "neg"
    if "pos" in lname:
        return "pos"
    if "zero" in lname:
        return "zero"
    return "unknown"

In [3]:
# Load clean dataset (as per lecturer's notebook, but using gctg-clean)
dataset = pm.Dataset("gctg-clean.yaml", str(BASE)).download().load()
# Keep dataset as-is (no split); we'll handle stimulus grouping later
print(f"Loaded gaze frames: {len(dataset.gaze)}")

INFO:pymovements.dataset.dataset:        You are downloading the gctg dataset. Please be aware that pymovements does not
        host or distribute any dataset resources and only provides a convenient interface to
        download the public dataset resources that were published by their respective authors.

        Please cite the referenced publication if you intend to use the dataset in your research.
        


Using already downloaded and verified file: data-clean\downloads\gctg-data-clean.zip
Extracting gctg-data-clean.zip to data-clean\raw


  0%|          | 0/822 [00:00<?, ?it/s]

  0%|          | 1/822 [00:00<03:18,  4.13it/s]

  0%|          | 2/822 [00:00<04:01,  3.39it/s]

...

 90%|████████▉ | 739/822 [00:04<00:00, 619.93it/s]

 98%|█████████▊| 806/822 [00:04<00:00, 634.06it/s]

100%|██████████| 822/822 [00:04<00:00, 190.21it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

Loaded gaze frames: 12


In [4]:
# Collect stimuli and exclude practice
all_samples = pl.concat([g.samples for g in dataset.gaze])
stimulus_names = all_samples["stimulus"].unique().to_list()
stimulus_names = [s for s in stimulus_names if not is_practice(s)]
subjects = all_samples["subject_id"].unique().to_list()
print(f"Stimuli (non-practice): {len(stimulus_names)} | Subjects: {len(subjects)} -> {sorted(subjects)}")
stimulus_names[:10]

Stimuli (non-practice): 152 | Subjects: 12 -> ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12']


['delayed-pos.interest',
 'prize-zero.text.0',
 'prize-neg.interest',
 'prize-pos.difficulty',
 'delayed-pos.text.2',
 'delayed-pos.naturalness',
 'goldfish-pos.interest',
 'delayed-zero.difficulty',
 'blackout-zero.question',
 'delayed-neg.question']

In [5]:
# Load word AOIs for each stimulus from the clean package
stimuli = {}
missing_aois = []
for stimulus_name in stimulus_names:
    aois_path = BASE / "raw" / "stimuli" / f"{stimulus_name}.word.csv"
    if not aois_path.exists():
        missing_aois.append(stimulus_name)
        continue
    aois = pl.read_csv(aois_path)
    stimulus = pm.stimulus.TextStimulus(
        aois,
        aoi_column="content",
        start_x_column="left",
        start_y_column="top",
        end_x_column="right",
        end_y_column="bottom",
    )
    stimuli[stimulus_name] = stimulus
print(f"Loaded AOIs for {len(stimuli)} stimuli. Missing AOIs: {len(missing_aois)}")

Loaded AOIs for 152 stimuli. Missing AOIs: 0


In [6]:
# Event detection using IDT defaults, then fixation locations in pixel space
# (matches lecturer’s pipeline: pix2deg -> detect("idt") -> compute_properties("location", pixel))
dataset.pix2deg()
dataset.detect("idt", clear=True)
dataset.compute_properties(("location", {"position_column": "pixel"}))
print("Events detected and fixation locations computed.")

  0%|          | 0/12 [00:00<?, ?it/s]

0it [00:00, ?it/s]

C:\Python312\Lib\site-packages\pymovements\events\detection\_idt.py:48: RuntimeWarning: All-NaN slice encountered
  return sum(np.nanmax(positions, axis=0) - np.nanmin(positions, axis=0))


  0%|          | 0/12 [00:00<?, ?it/s]

Events detected and fixation locations computed.


In [7]:
# Deprecated mapping approach (kept as no-op to maintain cell order)
# Mapping to AOIs is handled later via explicit per-subject-per-stimulus processing.
len([])

0

In [8]:
# Quick head (does not modify outputs)
trt.sample(n=min(5, trt.height)) if 'trt' in locals() else None

In [9]:
# Map fixations to word AOIs via point-in-rectangle and compute TRT reliably

def map_events_to_aois(events_df: pl.DataFrame, aoi_df: pl.DataFrame) -> pl.DataFrame:
    # events_df has 'location' as [x, y] list; expand to columns
    ev = events_df.with_columns([
        pl.col("location").list.first().alias("x"),
        pl.col("location").list.last().alias("y"),
    ])
    # Ensure AOI columns exist
    aois = aoi_df.select([
        pl.col("index"),
        pl.col("content"),
        pl.col("left"),
        pl.col("right"),
        pl.col("top"),
        pl.col("bottom"),
    ])
    # Cross-join and filter by within-rect; for performance we can prefilter by x/y bounds
    ev_small = ev.select(["subject_id", "stimulus", "duration", "x", "y"])  # keep necessary cols
    joined = ev_small.join(aois, how="cross")
    mapped = joined.filter(
        (pl.col("x") >= pl.col("left")) & (pl.col("x") <= pl.col("right")) &
        (pl.col("y") >= pl.col("top")) & (pl.col("y") <= pl.col("bottom"))
    )
    # Aggregate TRT by AOI
    trt = (
        mapped
        .group_by(["subject_id", "stimulus", "index", "content"]) 
        .agg(pl.col("duration").sum().alias("total_reading_time"))
    )
    # Right join to include AOIs with TRT=0
    # Use stimulus-wide AOIs; fill subject_id/stimulus later
    trt_full = (
        trt.join(aois.select(["index", "content"]), on=["index", "content"], how="right")
        .with_columns(pl.col("total_reading_time").fill_null(0))
    )
    # Fill subject and stimulus ids (unique within events)
    subj = events_df["subject_id"].unique().item()
    stim = events_df["stimulus"].unique().item()
    trt_full = trt_full.with_columns([
        pl.lit(subj).alias("subject_id").cast(pl.Utf8),
        pl.lit(stim).alias("stimulus").cast(pl.Utf8),
        pl.lit(extract_condition(stim)).alias("condition")
    ])
    return trt_full.select(["subject_id", "stimulus", "condition", "index", "content", "total_reading_time"])

In [10]:
# Corrected processing: iterate per subject and per stimulus within subject
trt_tables = []
cache_written = 0
for g in dataset.gaze:
    ev_all = g.events.frame
    subj = ev_all["subject_id"].unique().item()
    stims = ev_all["stimulus"].unique().to_list()
    for stim in stims:
        if is_practice(stim):
            continue
        if stim not in stimuli:
            continue
        sub_ev = ev_all.filter(pl.col("stimulus") == stim)
        # cache raw events with locations
        out_path = CACHE_DIR / f"events_{subj}_{stim}.parquet"
        sub_ev.write_parquet(out_path)
        cache_written += 1
        # compute TRT
        trt_tables.append(map_events_to_aois(sub_ev, stimuli[stim].aois))

print(f"Cached event tables: {cache_written}")
if trt_tables:
    trt = pl.concat(trt_tables, how="vertical_relaxed")
    out_csv = PROCESSED_DIR / "trt_by_word.csv"
    out_parquet = PROCESSED_DIR / "trt_by_word.parquet"
    trt.write_csv(out_csv)
    trt.write_parquet(out_parquet)
    print(f"Saved TRT rows: {trt.height} -> {out_csv}")
else:
    raise RuntimeError("No TRT tables produced. Verify events and AOIs.")

Cached event tables: 555
Saved TRT rows: 36160 -> data-clean\processed\trt_by_word.csv


In [11]:
# Preview a small sample of the final TRT table (non-destructive)
trt.head(10).to_pandas()

,subject_id,stimulus,condition,index,content,total_reading_time
0,P01,goldfish-pos.question,pos,0,What,0
1,P01,goldfish-pos.question,pos,1,role,253
2,P01,goldfish-pos.question,pos,2,does,217
3,P01,goldfish-pos.question,pos,3,Finley,0
4,P01,goldfish-pos.question,pos,4,the,0
5,P01,goldfish-pos.question,pos,5,goldfish,170
6,P01,goldfish-pos.question,pos,6,seem,121
7,P01,goldfish-pos.question,pos,7,to,0
8,P01,goldfish-pos.question,pos,8,play,0
9,P01,goldfish-pos.question,pos,9,in,0
